### Executive Dashboard: Hospital Financial & Readmission Overview
This interactive dashboard simulates the 'end product' for clinical and financial leadership.

In [7]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

# Load the dataset (assuming it's already uploaded or available)
df = pd.read_csv('/content/hospital_readmission_dataset.csv')

# Since the specific financial columns might be missing in the raw readmission file,
# we will use the available data and add simulated financial metrics for the demonstration.
# Check available columns: print(df.columns)

# Simulating financial columns if they don't exist for the dashboard demo
if 'billed_amount' not in df.columns:
    np.random.seed(42)
    df['billed_amount'] = np.random.uniform(5000, 50000, size=len(df))
    df['collected_amount'] = df['billed_amount'] * np.random.uniform(0.6, 0.9, size=len(df))
    df['denial_reason'] = np.random.choice(['Coding Error', 'Lack of Info', 'Not Covered', 'Duplicate Claim', 'Timely Filing'], size=len(df))
    df['payer'] = np.random.choice(['Medicare', 'Medicaid', 'Blue Cross', 'Aetna', 'UnitedHealth'], size=len(df))
    df['days_in_ar'] = np.random.randint(15, 90, size=len(df))

# 1. Financial Capture: Billed vs. Collected
billed_total = df['billed_amount'].sum()
collected_total = df['collected_amount'].sum()

fig1 = go.Figure(data=[
    go.Bar(name='Billed', x=['Total Revenue'], y=[billed_total], marker_color='#3498db'),
    go.Bar(name='Collected', x=['Total Revenue'], y=[collected_total], marker_color='#2ecc71')
])
fig1.update_layout(title='Total Billed vs. Total Collected', barmode='group', template='plotly_white', yaxis_title='Amount ($)')

# 2. Top 5 Denial Reasons
denials = df['denial_reason'].value_counts().nlargest(5).reset_index()
denials.columns = ['Reason', 'Count']

fig2 = px.pie(denials, values='Count', names='Reason', title='Top 5 Denial Reasons', hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdBu)

# 3. Days in A/R by Payer
ar_by_payer = df.groupby('payer')['days_in_ar'].mean().sort_values(ascending=False).reset_index()

fig3 = px.bar(ar_by_payer, x='payer', y='days_in_ar',
             title='Average Days in A/R by Payer',
             labels={'days_in_ar': 'Avg Days', 'payer': 'Insurance Provider'},
             color='days_in_ar', color_continuous_scale='Viridis')

# Show plots
fig1.show()
fig2.show()
fig3.show()

### Collection Rate Analysis by Payer
This metric shows the percentage of billed revenue actually collected for each insurance provider.

In [ ]:
# Calculate collection rate per payer
collection_stats = df.groupby('payer').agg({
    'billed_amount': 'sum',
    'collected_amount': 'sum'
}).reset_index()

collection_stats['collection_rate'] = (collection_stats['collected_amount'] / collection_stats['billed_amount']) * 100

# Visualize Collection Rate
fig4 = px.bar(collection_stats.sort_values('collection_rate', ascending=False),
             x='payer', y='collection_rate',
             title='Collection Rate (%) by Payer',
             labels={'collection_rate': 'Collection Rate (%)', 'payer': 'Insurance Provider'},
             text_auto='.2f',
             color='collection_rate', color_continuous_scale='Greens')

fig4.update_layout(yaxis_range=[0, 100], template='plotly_white')
fig4.show()

# Display the raw table
display(collection_stats[['payer', 'collection_rate']].sort_values(by='collection_rate', ascending=False))

,payer,collection_rate
4,UnitedHealth,75.288278
2,Medicaid,75.080057
1,Blue Cross,74.801355
3,Medicare,74.743662
0,Aetna,74.678554


### Payer Performance Comparison: Collection Rate vs. Days in A/R
This chart helps identify the relationship between payment speed and payment completion.

In [ ]:
# Merge the two metrics for comparison
comparison_df = pd.merge(collection_stats, ar_by_payer, on='payer')

# Create a grouped bar chart using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig5 = make_subplots(specs=[[{"secondary_y": True}]])

fig5.add_trace(
    go.Bar(x=comparison_df['payer'], y=comparison_df['collection_rate'], name="Collection Rate (%)", marker_color='#2ecc71'),
    secondary_y=False,
)

fig5.add_trace(
    go.Scatter(x=comparison_df['payer'], y=comparison_df['days_in_ar'], name="Avg Days in A/R", mode='lines+markers', marker_color='#e74c3c'),
    secondary_y=True,
)

fig5.update_layout(
    title_text="Collection Rate vs. Days in A/R by Payer",
    template='plotly_white'
)

fig5.update_yaxes(title_text="Collection Rate (%)", secondary_y=False, range=[0, 100])
fig5.update_yaxes(title_text="Average Days in A/R", secondary_y=True)

fig5.show()

### Deep Dive: Denial Analysis for Lowest Performing Payer
Identifying the specific bottlenecks for the provider with the lowest collection rate.

In [ ]:
# Identify the lowest performing payer
lowest_payer = collection_stats.sort_values('collection_rate').iloc[0]['payer']

# Filter data for this payer
lowest_payer_denials = df[df['payer'] == lowest_payer]['denial_reason'].value_counts().reset_index()
lowest_payer_denials.columns = ['Reason', 'Count']

# Visualize
fig6 = px.bar(lowest_payer_denials,
             x='Reason', y='Count',
             title=f'Denial Reason Distribution for {lowest_payer}',
             labels={'Count': 'Number of Denials', 'Reason': 'Denial Category'},
             color='Count', color_continuous_scale='Reds')

fig6.update_layout(template='plotly_white')
fig6.show()

print(f"The lowest performing payer is {lowest_payer}.")

The lowest performing payer is Aetna.


### Portfolio Feature: Embedding
To embed this in a portfolio website, you can export these Plotly charts as HTML files:
```python
fig1.write_html('billed_vs_collected.html')
```
Alternatively, for a live Looker Studio dashboard, you would:
1. Upload the `cleaned_hospital_data.csv` to Google Sheets or BigQuery.
2. Open [Looker Studio](https://lookerstudio.google.com/).
3. Select 'Data Source' -> Google Sheets.
4. Drag and drop these metrics into the canvas.